# Assignment 3: Transformer is All You Need

**Task:** Train a small decoder-only Transformer for next-token prediction on Tiny Shakespeare, then visualize its attention and explain its learning behavior.

**Track:** learning · **Step 0** — setup and data loading

This track mirrors the submission notebook and adds learning notes, self-check questions, and report prompts. The submission track stays clean for grading.

1. Data preparation (positional split, subword tokenizer, token windows)
2. A tiny causal Transformer built from basic PyTorch layers
3. Training, loss curves, perplexity, and attention visualization
4. A controlled comparison of configurations


## 0. Setup

Import the libraries used below and fix the constants that make the work reproducible. The hardware and package versions are printed because training time and exact losses depend on them.


**Learning notes — Setup**

- `SEED = 42` makes the RNG streams for Python, NumPy, and PyTorch agree so two runs can be compared.
- Print the hardware and package versions: training time and the exact loss depend on them, so the report must record what produced the numbers.
- Prefer GPU in Colab when available, but every reported figure must say which device was used.

**Self-check**

1. Why does changing the seed change the training curves even with the same split and tokenizer?
2. What would go wrong if one configuration ran on GPU and another on CPU and you compared their wall-clock times as if hardware were identical?


In [ ]:
import hashlib
import os
import platform
import random
from datetime import date
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
import numpy as np
import tokenizers
import torch

SEED = 42
TRAIN_FRACTION = 0.8


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python      {platform.python_version()}")
print(f"PyTorch     {torch.__version__}")
print(f"tokenizers  {tokenizers.__version__}")
print(f"NumPy       {np.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"Platform    {platform.platform()}")
print(f"CPU cores   {os.cpu_count()}")
print(f"Device      {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"Seed        {SEED}")


### 0.1 Load the data

Prefer a local `input.txt` in the notebook working directory (Colab upload). If it is missing, download it from the source: Andrej Karpathy's char-rnn repository. The SHA-256 check guarantees that every run and every experiment uses exactly the same text.


**Learning notes — Data loading**

- The handout requires recording the **source** and **download date**. The SHA-256 check is how we prove every experiment used the same file.
- Prefer a Colab upload of `input.txt` (same pattern as Assignments 1 and 2). The GitHub raw URL is only the fallback.
- Tiny Shakespeare is a teaching corpus: scores here are not evidence of general language modeling ability.

**Self-check**

1. Why split the raw character text by position *before* fitting a tokenizer?
2. If the SHA-256 check failed, what would you do before continuing?

**Report prompt**

In the Methods section, state the source URL, download date, character count, and that the same file (checksum) was used for every configuration.


In [ ]:
DATA_FILE = Path("input.txt")
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
EXPECTED_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"

if DATA_FILE.exists():
    # The uploaded file is the repository copy, fetched from DATA_URL on this date.
    download_date = "2026-11-21"
else:
    urlretrieve(DATA_URL, DATA_FILE)
    download_date = date.today().isoformat()

raw_bytes = DATA_FILE.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()
assert sha256 == EXPECTED_SHA256, f"Unexpected file contents (SHA-256 {sha256})"
text = raw_bytes.decode("utf-8")

print(f"Source         {DATA_URL}")
print(f"Download date  {download_date}")
print(f"SHA-256        {sha256}")
print(f"Characters     {len(text):,}")
print(f"Lines          {text.count(chr(10)):,}")
print(f"Unique chars   {len(set(text))}")
print()
print(text[:250])


## 1. Data preparation

The raw text is split by position before any tokenizer is fitted or any window is created, so nothing learned from validation text can reach training.


**Coming in Step 1**

- Split first 80% / final 20% by character position
- Fit BPE (vocabulary ≤ 500) on training text only; explain unseen symbols
- Build `N+1` windows: input `z[0:N]`, target `z[1:N+1]`
- Training windows may overlap; validation windows use stride `N`; no window crosses the split

**Self-check (preview)**

1. Why are the targets shifted by one token?
2. What would leak if a window straddled the train/validation cut?


## 2. Tiny decoder-only Transformer

The model is built from token embeddings, sinusoidal positional encodings, causal multi-head self-attention written with tensor operations, RMSNorm, residual connections, a feed-forward network, and a vocabulary projection.


**Coming in Step 2**

- Token embedding + sinusoidal positions (or RoPE on Q/K)
- Causal multi-head attention from basic layers (`nn.Linear` / `nn.Embedding` / `nn.RMSNorm` only for those parts)
- Causality checks in `eval()`: future attention weights are zero; changing future tokens leaves earlier logits unchanged
- Cross-entropy on raw logits with shifted targets; one controlled change under a fixed step budget

**Self-check (preview)**

1. What prevents a token from seeing the future?
2. Why must softmax *not* be applied before `cross_entropy`?


## 3. Training curves, perplexity, and attention

All evaluation runs in `model.eval()` under `torch.no_grad()`. Validation loss is the total negative log-likelihood divided by the number of scored target tokens, and perplexity is its exponential.


**Coming in Step 3**

- Train and validation cross-entropy vs steps
- Validation loss = total NLL ÷ scored target tokens; `PPL = exp(loss)` (never average batch PPLs)
- Attention heatmaps for ≥ 2 heads or layers on one fixed validation sequence, labeled with real tokens

**Self-check (preview)**

1. Why is averaging batch perplexities wrong?
2. Why does a bright attention cell alone not prove causal importance?


## 4. Comparison of configurations

The configurations differ in one controlled setting; the split, tokenizer, context length, seed, and training-step budget are fixed.


**Coming in Step 4**

- Compact table: configurations, parameter counts, training time, best validation loss, PPL
- Discuss what the controlled change did, what the maps showed, positional encoding and masking, runtime bottlenecks, and limitations
